In [1]:
#Step 1 — Create Promotion Analysis Notebook
import pandas as pd

df = pd.read_csv("../data/raw/train.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [2]:
#Step 2 — Check Promotion vs Non-Promotion Sales
promo_analysis = df.groupby("onpromotion")["sales"].mean()

promo_analysis

onpromotion
0       158.246681
1       467.556532
2       662.925632
3       871.408092
4       969.916135
          ...     
719    6681.000000
720    6154.000000
722    5846.000000
726    6044.000000
741    7517.000000
Name: sales, Length: 362, dtype: float64

In [3]:
#Step 3 — Calculate Promotion Lift
no_promo = promo_analysis[0]
promo = promo_analysis[1]

lift = (promo - no_promo) / no_promo

print("Promotion lift:", lift)

Promotion lift: 1.9546056061654817


In [4]:
#Step 4 — Promotion Impact by Product Family
family_promo = df.groupby(["family","onpromotion"])["sales"].mean().unstack()

family_promo.head()

onpromotion,0,1,2,3,4,5,6,7,8,9,...,702,710,716,717,718,719,720,722,726,741
family,,,,,,,,,,,,,,,,,,,,,
AUTOMOTIVE,5.853087,12.271947,13.480392,15.923954,18.936170,20.728814,20.428571,21.000000,19.250000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BABY CARE,0.109624,1.660377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BEAUTY,3.078452,8.079021,9.556841,6.837037,9.154762,13.333333,18.250000,18.000000,60.500000,61.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BEVERAGES,1292.272591,2254.404076,2567.288278,2950.067889,3080.714092,3054.160355,3086.897638,3098.250751,3157.090766,3103.272506,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BOOKS,0.070797,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
family_promo["promo_lift"] = (
    (family_promo[1] - family_promo[0]) / family_promo[0]
)

family_promo = family_promo.sort_values("promo_lift", ascending=False)

family_promo.head(10)

onpromotion,0,1,2,3,4,5,6,7,8,9,...,710,716,717,718,719,720,722,726,741,promo_lift
family,,,,,,,,,,,,,,,,,,,,,
SCHOOL AND OFFICE SUPPLIES,1.022018,17.597816,19.641388,21.577815,26.423767,37.440994,49.414286,69.133333,87.365217,94.200000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.218689
BABY CARE,0.109624,1.660377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.146048
PET SUPPLIES,3.507379,15.852740,20.433333,4.074074,5.916667,9.000000,8.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.519825
HOME AND KITCHEN II,10.275239,36.150215,46.378895,48.828916,51.059920,57.053299,70.245000,67.811765,47.000000,42.466667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.518187
HOME APPLIANCES,0.456876,1.396552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.056740
HOME CARE,101.956009,309.011856,302.997940,269.573680,265.446055,284.398529,298.128109,296.849125,300.920043,314.924674,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.030835
PRODUCE,792.101911,2103.026708,1633.097879,1939.587704,2099.289322,2310.249790,2444.039980,2372.915170,2836.096994,3586.639565,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.654995
PLAYERS AND ELECTRONICS,6.081521,16.062069,17.188679,10.476190,11.055556,11.590909,15.882353,13.636364,22.285714,28.285714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.641127
BEAUTY,3.078452,8.079021,9.556841,6.837037,9.154762,13.333333,18.250000,18.000000,60.500000,61.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.624377


In [6]:
#Step 5 — Save Promotion Impact Table
family_promo.to_csv("../data/processed/promotion_lift.csv")

In [7]:
#Step 6 — Update Simulation Engine
import pandas as pd

promo_lift = pd.read_csv("../data/processed/promotion_lift.csv")

In [8]:
#Step 7 — Data-Driven Simulation Function
def simulate_promotion(base_demand, family, promotion):

    if promotion == 0:
        return base_demand
    
    lift = promo_dict.get(family, 0.15)
    
    demand = base_demand * (1 + lift)
    
    return demand